# LAB 27 — Алгоритми Сортування: NYC Taxi Dispatch Optimizer

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Що ми будуємо?

Ми будуємо **оптимізатор диспетчеризації таксі** на реальних даних NYC Yellow Taxi.
Задача: отримати черговий запит на поїздку і **відсортувати** незайняті машини так, щоб водій з найближчої зони приїхав першим.

**Центральне питання цієї лабораторії:**
> Коли реалізовувати власний алгоритм сортування? Коли використовувати `sorted()`?
> Чому всі алгоритми «$O(n \log n)$» мають абсолютно різну швидкість на практиці?

---

## Карта лабораторії

| Частина | Тема | Складність |
|---------|------|------------|
| 1 | Завантаження та підготовка даних (DuckDB) | Архітектура |
| 2 | Python `sorted()` — правильний старт | $O(n \log n)$ |
| 3 | Insertion Sort — чемпіон малих масивів | $O(n)$ best case |
| 4 | Merge Sort — гарантія і стабільність | $O(n \log n)$ |
| 5 | Quick Sort — in-place на великих даних | $O(n \log n)$ avg |
| 6 | Bubble Sort — наочна катастрофа | $O(n^2)$ |
| 7 | Benchmark: всі алгоритми на реальних даних | Вимірювання |
| 8 | Dispatch Optimizer — практичний кейс | Реалізм |
| 9 | Аналітика: топ зони, розподіл тарифів | Візуалізація |
| 10 | Фінальний інсайт | Архітектура |

---

> **Принцип:** Сортування — це не просто порядок. Це рішення про те,
> як ваш CPU взаємодіє з пам'яттю. Вибір алгоритму — це вибір між
> мікросекундами та хвилинами очікування.

---

## Частина 1: Завантаження Даних — DuckDB Lazy Architecture

---

### Архітектура шарів

```
NYC Parquet (3M+ рядків, хмара)
        ↓  HTTPS + httpfs extension
DuckDB VIEW «trips»   ← визначає ЯК читати, але не читає нічого
        ↓  SQL SAMPLE {n} ROWS
pandas DataFrame      ← лише n рядків у RAM (~5 000)
        ↓  .to_dict('records')
Python list[dict]     ← наші «поїздки» для сортування
```

**Колонки, які нас цікавлять:**

| Колонка | Тип | Призначення |
|---------|-----|-------------|
| `pickup_dt` | datetime | Час виклику — для часткового сортування |
| `trip_distance` | float | Відстань поїздки (милі) |
| `fare_amount` | float | Базовий тариф (USD) |
| `tip_amount` | float | Чайові (USD) |
| `total_amount` | float | Загальна сума (USD) |
| `pu_zone` | int | Зона посадки (1–263) |
| `do_zone` | int | Зона висадки (1–263) |
| `passengers` | int | Кількість пасажирів |

---

In [ ]:
# ── Залежності ─────────────────────────────────────────────────────────────
# pip install duckdb pandas matplotlib ipywidgets

import time
import timeit
import random
from collections import defaultdict

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── DuckDB: підключення в пам'яті ─────────────────────────────────────────
con = duckdb.connect(database=":memory:")

try:
    con.execute("INSTALL httpfs; LOAD httpfs;")
    print("httpfs завантажено")
except Exception as e:
    print(f"  httpfs вже присутній: {e}")

# ── Lazy VIEW ─────────────────────────────────────────────────────────────
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2023-01.parquet"
)

con.execute(f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT
        tpep_pickup_datetime                     AS pickup_dt,
        CAST(trip_distance   AS DOUBLE)          AS trip_distance,
        CAST(fare_amount     AS DOUBLE)          AS fare_amount,
        CAST(tip_amount      AS DOUBLE)          AS tip_amount,
        CAST(total_amount    AS DOUBLE)          AS total_amount,
        CAST(PULocationID    AS INTEGER)         AS pu_zone,
        CAST(DOLocationID    AS INTEGER)         AS do_zone,
        CAST(passenger_count AS INTEGER)         AS passengers
    FROM read_parquet('{DATA_URL}')
    WHERE
        trip_distance > 0.1 AND trip_distance < 100
        AND fare_amount   > 0 AND fare_amount   < 500
        AND total_amount  > 0
        AND passenger_count >= 1
""")

print("VIEW 'trips' зареєстровано (lazy - жодних даних ще в RAM)")

In [ ]:
# ── SQL SAMPLE: завантажуємо 5 000 рядків ─────────────────────────────────
SAMPLE_SIZE = 5_000

print(f"Виконуємо SQL SAMPLE {SAMPLE_SIZE:,} ROWS...")
t0 = time.perf_counter()

df = con.execute(f"""
    SELECT *
    FROM trips
    USING SAMPLE {SAMPLE_SIZE} ROWS
    ORDER BY pickup_dt
""").df().reset_index(drop=True)

# ── Конвертуємо в list[dict] — один dict = одна поїздка ──────────────────
trips = df.to_dict('records')

t1 = time.perf_counter()

print(f"Завантажено за {(t1-t0)*1000:.0f} мс")
print(f"  Рядків у пам'яті : {len(trips):,}  (з 3M+ у хмарі)")
print(f"  RAM estimate     : ~{df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()
print("Перші 3 поїздки:")
for i, t in enumerate(trips[:3]):
    print(f"  [{i}] dist={t['trip_distance']:.1f}mi  fare=${t['fare_amount']:.2f}"
          f"  tip=${t['tip_amount']:.2f}  zone={t['pu_zone']}→{t['do_zone']}")

In [ ]:
# ── Статистика повного датасету через DuckDB (не завантажуючи в Python) ────
stats = con.execute("""
    SELECT
        COUNT(*)                         AS total_trips,
        ROUND(AVG(trip_distance), 2)     AS avg_distance_mi,
        ROUND(AVG(fare_amount), 2)       AS avg_fare_usd,
        ROUND(AVG(tip_amount), 2)        AS avg_tip_usd,
        ROUND(AVG(total_amount), 2)      AS avg_total_usd,
        COUNT(DISTINCT pu_zone)          AS unique_pu_zones,
        MAX(trip_distance)               AS max_distance_mi
    FROM trips
""").fetchone()

print("=== Статистика ПОВНОГО датасету (обчислено в DuckDB, не в Python) ===")
labels = ["Всього поїздок", "Середня відстань (mi)", "Середній тариф ($)",
          "Середні чайові ($)", "Середня сума ($)", "Унікальних зон", "Макс. відстань (mi)"]
for label, val in zip(labels, stats):
    print(f"  {label:<25}: {val:>12,}" if isinstance(val, int) else
          f"  {label:<25}: {val:>12.2f}")
print()
print(f"  Наша вибірка: {len(trips):,} рядків з {stats[0]:,} (={len(trips)/stats[0]*100:.2f}%)")

---

## Частина 2: Python `sorted()` — Правильний Старт

---

### Золоте правило

> **Завжди починайте з `sorted()` або `list.sort()`.** Тільки якщо є конкретна причина (embedded, контроль пам'яті, навчання) — реалізовуйте власний алгоритм.

### Що вміє `sorted()` / `list.sort()`

```python
# Одне поле
sorted(trips, key=lambda t: t['trip_distance'])

# Кілька полів: спочатку за зоною, потім за відстанню
sorted(trips, key=lambda t: (t['pu_zone'], t['trip_distance']))

# Зворотний порядок для одного поля
sorted(trips, key=lambda t: t['total_amount'], reverse=True)

# Мішаний: зона asc, сума desc (негативне значення для desc)
sorted(trips, key=lambda t: (t['pu_zone'], -t['total_amount']))
```

---

In [ ]:
# ── Сортування через Python built-in (Timsort) ────────────────────────────

print("=== sorted() / list.sort() — Timsort під капотом ===")

# 1. Сортування за відстанню (ascending)
by_distance = sorted(trips, key=lambda t: t['trip_distance'])
print("\n1. Топ-3 НАЙКОРОТШИХ поїздки:")
for t in by_distance[:3]:
    print(f"   dist={t['trip_distance']:.2f}mi  fare=${t['fare_amount']:.2f}  zone {t['pu_zone']}→{t['do_zone']}")

print("\n   Топ-3 НАЙДОВШИХ поїздки:")
for t in by_distance[-3:]:
    print(f"   dist={t['trip_distance']:.2f}mi  fare=${t['fare_amount']:.2f}  zone {t['pu_zone']}→{t['do_zone']}")

# 2. Сортування за сумою (descending) — найдорожчі поїздки
by_total_desc = sorted(trips, key=lambda t: t['total_amount'], reverse=True)
print("\n2. Топ-3 НАЙДОРОЖЧИХ поїздки:")
for t in by_total_desc[:3]:
    print(f"   total=${t['total_amount']:.2f}  dist={t['trip_distance']:.1f}mi  tip=${t['tip_amount']:.2f}")

# 3. Мульти-ключ: за зоною посадки, потім за відстанню (dispatch optimizer)
dispatch_order = sorted(trips, key=lambda t: (t['pu_zone'], t['trip_distance']))
print("\n3. Dispatch order (зона ASC, відстань ASC) — перші 5:")
for t in dispatch_order[:5]:
    print(f"   zone={t['pu_zone']:>3}  dist={t['trip_distance']:.2f}mi  fare=${t['fare_amount']:.2f}")

# 4. Замір часу
t_timsort = timeit.timeit(
    lambda: sorted(trips, key=lambda t: (t['pu_zone'], t['trip_distance'])),
    number=100
) / 100 * 1000
print(f"\n4. Час sorted() на {len(trips):,} поїздках: {t_timsort:.3f} мс")
print(f"   Це наш baseline — будь-яка власна реалізація буде повільніша!")

---

## Частина 3: Insertion Sort — Чемпіон Малих Масивів

---

### Сценарій: Dispatch mini-batch

Диспетчер отримує нові запити малими пачками (5–20 поїздок за раз). Для таких мікро-груп Insertion Sort обходить навіть Timsort — бо не має накладних витрат рекурсії і ідеально покриває одну кеш-лінію CPU.

### Чому Insertion Sort переважає для n < 20?

```
Timsort для n=8:
  - Перевірити: чи є runs
  - Перевірити: чи потрібен merge
  - Викликати Insertion Sort (внутрішньо!)
  Overhead: ~50 ns на виклики

Прямий Insertion Sort для n=8:
  - Зразу порівнювати сусідів
  - 8 елементів = 32 байти = 1 кеш-лінія (64B)
  Overhead: нуль

Timsort САМ використовує Insertion Sort для малих runs.
```

---

In [ ]:
# ── Insertion Sort для диспетчера ─────────────────────────────────────────

def insertion_sort_trips(trips_batch, key_fn):
    """
    Insertion Sort для малих масивів поїздок.
    key_fn: функція що повертає значення для порівняння.
    O(n^2) worst case, але O(n) якщо майже відсортовано.
    Ідеально для пачок 5-20 поїздок.
    """
    arr = list(trips_batch)       # копія — не мутуємо оригінал
    for i in range(1, len(arr)):
        current = arr[i]
        current_key = key_fn(current)
        pos = i
        # Зсуваємо більші елементи вправо, шукаємо позицію для current
        while pos > 0 and key_fn(arr[pos - 1]) > current_key:
            arr[pos] = arr[pos - 1]   # memory shift: зсув без swap!
            pos -= 1
        arr[pos] = current            # вставляємо на правильну позицію
    return arr


# ── Демонстрація: мікро-батч від диспетчера ──────────────────────────────
BATCH_SIZE = 10
mini_batch = trips[:BATCH_SIZE]

print(f"=== Insertion Sort: мікро-батч {BATCH_SIZE} поїздок ===")
print("\nДо сортування:")
for t in mini_batch:
    print(f"  zone={t['pu_zone']:>3}  dist={t['trip_distance']:.2f}mi")

sorted_batch = insertion_sort_trips(mini_batch, key_fn=lambda t: t['trip_distance'])
print("\nПісля Insertion Sort (за trip_distance):")
for t in sorted_batch:
    print(f"  zone={t['pu_zone']:>3}  dist={t['trip_distance']:.2f}mi")

# ── Benchmark: Insertion Sort vs Timsort для малих n ────────────────────
print()
print("=== Benchmark: малі масиви (мікросекунди) ===")
print(f"{'n':>6} | {'Insertion Sort':>16} | {'Timsort sorted()':>17} | {'Winner':>8}")
print("-" * 58)

REPS = 50_000
key_fn = lambda t: t['trip_distance']

for n in [5, 10, 15, 20, 50, 100]:
    batch = (trips * (n // len(trips) + 1))[:n]
    t_ins = timeit.timeit(
        lambda b=batch: insertion_sort_trips(b, key_fn),
        number=REPS
    ) / REPS * 1e6
    t_tim = timeit.timeit(
        lambda b=batch: sorted(b, key=key_fn),
        number=REPS
    ) / REPS * 1e6
    winner = "Insertion" if t_ins < t_tim else "Timsort"
    print(f"{n:>6} | {t_ins:>14.2f} мкс | {t_tim:>15.2f} мкс | {winner:>8}")

print()
print("Висновок: Timsort SAM використовує Insertion Sort для n < ~64")

---

## Частина 4: Merge Sort — Стабільність та Гарантія

---

### Сценарій: Звіт диспетчера за зміну

Менеджер хоче звіт: поїздки відсортовані **спочатку за зоною посадки, потім за сумою**. При цьому поїздки однієї зони мають **зберегти хронологічний порядок** (час посадки) — класична задача стабільного багаторівневого сортування.

### Чому Merge Sort для цього?

1. **Стабільний:** однакові ключі зберігають відносний порядок з вхідного масиву
2. **Гарантований $O(n \log n)$** — ніякого $O(n^2)$ при відсортованих даних
3. **Linked Lists:** злиття вказівниками без копіювання — $O(1)$ пам'яті

---

In [ ]:
# ── Merge Sort: реалізація та демонстрація стабільності ───────────────────

def merge(left, right, key_fn):
    """
    Злиття двох відсортованих списків.
    Використовує <= для забезпечення СТАБІЛЬНОСТІ:
    при рівних ключах лівий елемент (раніший) іде першим.
    Виділяє новий список -- O(n) пам'яті.
    """
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        # <= гарантує стабільність: рівні елементи зберігають оригінальний порядок
        if key_fn(left[i]) <= key_fn(right[j]):
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort_trips(arr, key_fn):
    """
    Merge Sort для списку поїздок.
    Рекурсивно ділимо навпіл, зливаємо з збереженням порядку рівних ключів.
    Повертає новий відсортований список (не in-place).
    """
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left  = merge_sort_trips(arr[:mid],  key_fn)  # ліва половина
    right = merge_sort_trips(arr[mid:],  key_fn)  # права половина
    return merge(left, right, key_fn)              # зливаємо відсортовані


# ── Демонстрація стабільності ─────────────────────────────────────────────
print("=== Демонстрація стабільності Merge Sort ===")
print()

# Беремо 15 поїздок із 3 зон, кожна вже в хронологічному порядку (pickup_dt asc)
zone_sample = [t for t in trips if t['pu_zone'] in (132, 161, 237)][:15]
if not zone_sample:
    zone_sample = trips[:15]

print("Вхід (хронологічний порядок всередині кожної зони):")
for i, t in enumerate(zone_sample):
    print(f"  [{i:>2}] zone={t['pu_zone']:>3}  dt={str(t['pickup_dt'])[:16]}  dist={t['trip_distance']:.1f}mi")

# Сортуємо за зоною — хронологічний порядок всередині зони має зберегтись
sorted_by_zone = merge_sort_trips(zone_sample, key_fn=lambda t: t['pu_zone'])

print()
print("Після Merge Sort за зоною (стабільний — порядок за dt всередині зони збережено):")
for i, t in enumerate(sorted_by_zone):
    print(f"  [{i:>2}] zone={t['pu_zone']:>3}  dt={str(t['pickup_dt'])[:16]}  dist={t['trip_distance']:.1f}mi")

# Верифікація: перевіряємо що всередині кожної зони dt монотонно зростає
zones_in_result = {}
stable = True
for t in sorted_by_zone:
    z = t['pu_zone']
    if z not in zones_in_result:
        zones_in_result[z] = t['pickup_dt']
    else:
        if t['pickup_dt'] < zones_in_result[z]:
            stable = False
            break
        zones_in_result[z] = t['pickup_dt']

print()
print(f"  Стабільність перевірена: {'OK (порядок dt збережено)' if stable else 'ПОРУШЕНО!'}")

---

## Частина 5: Quick Sort — In-Place на Великих Даних

---

### Сценарій: Ранжування водіїв за виручкою

В кінці зміни менеджер хоче отримати рейтинг поїздок за `total_amount` (від більшого до меншого). Дані великі — не хочемо виділяти $O(n)$ додаткової пам'яті як Merge Sort.

### In-place: чому важливо?

```
n = 50,000 поїздок × 200 байт/поїздка = 10 MB

Merge Sort: виділяє ще 10 MB тимчасових масивів = 20 MB total
Quick Sort: лише ~log(50000) * 8 байт на стек = ~128 байт

На embedded пристрої або мікросервісі з обмеженою RAM — критично!
```

---

In [ ]:
# ── Quick Sort in-place для поїздок ───────────────────────────────────────

def partition(arr, low, high, key_fn):
    """
    Lomuto partition: pivot = arr[high].
    Всі елементи < pivot -- зліва, > pivot -- справа.
    In-place swap — нова пам'ять не виділяється.
    """
    pivot_key = key_fn(arr[high])   # ключ опорного елемента
    i = low - 1                     # права межа «малих» елементів
    for j in range(low, high):
        if key_fn(arr[j]) <= pivot_key:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]   # in-place swap двох dict-посилань
    arr[i + 1], arr[high] = arr[high], arr[i + 1]  # pivot на своє місце
    return i + 1


def _quick_sort_recursive(arr, low, high, key_fn):
    """Рекурсивна частина Quick Sort. Не викликати напряму."""
    if low < high:
        pi = partition(arr, low, high, key_fn)
        _quick_sort_recursive(arr, low, pi - 1, key_fn)    # ліва частина
        _quick_sort_recursive(arr, pi + 1, high, key_fn)   # права частина


def quick_sort_trips(arr, key_fn, reverse=False):
    """
    Quick Sort (in-place) для списку поїздок.
    Сортує arr на місці та повертає його (для зручності).
    reverse=True: після сортування розвертаємо масив.
    """
    arr_copy = list(arr)   # копія — не мутуємо оригінал
    _quick_sort_recursive(arr_copy, 0, len(arr_copy) - 1, key_fn)
    if reverse:
        arr_copy.reverse()  # O(n) — але лише один прохід по пам'яті
    return arr_copy


# ── Демонстрація: рейтинг поїздок за total_amount ────────────────────────
print("=== Quick Sort: топ поїздки за total_amount ===")

key_total = lambda t: t['total_amount']
ranked = quick_sort_trips(trips, key_fn=key_total, reverse=True)

print("\nТОП-5 найдорожчих поїздок (Quick Sort, descending):")
for i, t in enumerate(ranked[:5]):
    print(f"  #{i+1}  total=${t['total_amount']:.2f}  "
          f"fare=${t['fare_amount']:.2f}  tip=${t['tip_amount']:.2f}  "
          f"dist={t['trip_distance']:.1f}mi  zone {t['pu_zone']}")

print("\nТОП-5 найдешевших поїздок:")
for i, t in enumerate(ranked[-5:][::-1]):
    print(f"  #{i+1}  total=${t['total_amount']:.2f}  "
          f"fare=${t['fare_amount']:.2f}  dist={t['trip_distance']:.2f}mi")

# Перевірка коректності
print()
timsort_result = sorted(trips, key=key_total, reverse=True)
match = [t['total_amount'] for t in ranked] == [t['total_amount'] for t in timsort_result]
print(f"  Результат збігається з sorted(): {'OK' if match else 'ПОМИЛКА'}")

---

## Частина 6: Bubble Sort — Наочна Катастрофа

---

### Чому ми все одно показуємо Bubble Sort?

Barack Obama на інтерв'ю Google: *"Bubble Sort is the wrong way to go."*
Але інженер повинен **розуміти чому** — не просто знати що це погано.

```
n=5,000 поїздок, Bubble Sort:
  Прохід 1:  5000 порівнянь + до 5000 swap
  Прохід 2:  4999 порівнянь + до 4999 swap
  ...
  Всього: ~12.5 мільйонів операцій swap
  Кожен swap = 3 операції запису в RAM
  = 37.5 мільйонів записів!

Timsort:
  ~5000 * log2(5000) ≈ 62,000 операцій
  600x менше!
```

---

In [ ]:
# ── Bubble Sort: оптимізована версія з підрахунком операцій ──────────────

def bubble_sort_trips(arr, key_fn):
    """
    Bubble Sort з оптимізацією (прапорець swapped).
    Повертає (sorted_arr, comparisons, swaps) для демонстрації.
    """
    arr = list(arr)
    n = len(arr)
    comparisons = 0
    swaps = 0

    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):   # -i: кінець вже відсортований
            comparisons += 1
            if key_fn(arr[j]) > key_fn(arr[j + 1]):
                arr[j], arr[j + 1] = arr[j + 1], arr[j]   # дорогий swap!
                swaps += 1
                swapped = True
        if not swapped:               # жодного swap -- масив відсортований
            break

    return arr, comparisons, swaps


# ── Демонстрація на малій вибірці ─────────────────────────────────────────
DEMO_SIZE = 200    # Bubble Sort надто повільний для 5000 в демо режимі
demo_batch = trips[:DEMO_SIZE]
key_dist = lambda t: t['trip_distance']

print(f"=== Bubble Sort на {DEMO_SIZE} поїздках ===")
t0 = time.perf_counter()
sorted_bubble, comparisons, swaps = bubble_sort_trips(demo_batch, key_dist)
t_bubble = time.perf_counter() - t0

t_timsort_demo = timeit.timeit(
    lambda: sorted(demo_batch, key=key_dist),
    number=1000
) / 1000

print(f"  Порівнянь (comparisons): {comparisons:,}")
print(f"  Замін (swaps):           {swaps:,}")
print(f"  Теоретичний O(n^2):      {DEMO_SIZE*(DEMO_SIZE-1)//2:,}")
print()
print(f"  Bubble Sort час:  {t_bubble*1000:.3f} мс")
print(f"  Timsort час:      {t_timsort_demo*1000:.3f} мс")
print(f"  Timsort швидше:   {t_bubble/t_timsort_demo:.1f}x")
print()

# Результат коректний?
timsort_check = sorted(demo_batch, key=key_dist)
correct = [t['trip_distance'] for t in sorted_bubble] == \
          [t['trip_distance'] for t in timsort_check]
print(f"  Результат коректний: {'OK' if correct else 'ПОМИЛКА'}")

# Екстраполяція на 5000 елементів
import math
ratio = (5000/DEMO_SIZE)**2   # квадратична залежність
print()
print(f"=== Екстраполяція: що буде при n=5000? ===")
print(f"  Estimated Bubble Sort: {t_bubble * ratio * 1000:.0f} мс")
print(f"  Timsort залишається:   ~{t_timsort_demo * (5000/DEMO_SIZE) * 1000:.2f} мс (лінійне масштабування)")

---

## Частина 7: Benchmark — Всі Алгоритми на Реальних Даних

---

Відтворюємо таблицю з реального академічного бенчмарку, але на справжніх даних NYC Taxi:

| Run | Bubble | Insertion | Merge | Quick | Timsort |
|-----|--------|-----------|-------|-------|---------|
| Avg | ~5s | ~1.6s | ~0.03s | ~0.017s | ~0.005s |

*(При n=5000 на Python, оригінальна стаття Stanford)*

---

In [ ]:
# ── Benchmark: всі алгоритми на реальних поїздках ─────────────────────────
import timeit
import random

# Ключ для сортування: відстань поїздки
KEY = lambda t: t['trip_distance']

# Розміри тестових вибірок
BENCH_SIZES = [500, 1_000, 2_000, 5_000]
RUNS = 3

def make_sample(n):
    """Генерує перемішану вибірку n поїздок."""
    pool = trips * (n // len(trips) + 1)
    sample = pool[:n]
    random.shuffle(sample)   # перемішуємо для fair benchmark
    return sample

# Обгортки що повертають результат (для перевірки коректності)
def run_bubble(data):     return bubble_sort_trips(data, KEY)[0]
def run_insertion(data):  return insertion_sort_trips(data, KEY)
def run_merge(data):      return merge_sort_trips(data, KEY)
def run_quick(data):      return quick_sort_trips(data, KEY)
def run_timsort(data):    return sorted(data, key=KEY)

algorithms = [
    ('Bubble Sort',    run_bubble),
    ('Insertion Sort', run_insertion),
    ('Merge Sort',     run_merge),
    ('Quick Sort',     run_quick),
    ('Timsort',        run_timsort),
]

bench_results = {name: [] for name, _ in algorithms}
bench_results['n'] = BENCH_SIZES

print(f"Benchmark на реальних даних NYC Taxi | ключ: trip_distance")
print(f"{'='*72}")
header = f"{'Алгоритм':<18}" + "".join(f" | {n:>8,}" for n in BENCH_SIZES) + " (мс)"
print(header)
print("-" * 72)

for name, func in algorithms:
    row = []
    for n in BENCH_SIZES:
        # Для Bubble на великих n пропускаємо (занадто повільно)
        if name == 'Bubble Sort' and n > 2_000:
            row.append(float('nan'))
            continue
        sample = make_sample(n)
        times = []
        for _ in range(RUNS):
            t = timeit.timeit(lambda s=sample: func(list(s)), number=1)
            times.append(t)
        avg = min(times) * 1000   # беремо мінімум (best performance)
        row.append(avg)
    bench_results[name] = row

    row_str = f"{name:<18}" + "".join(
        f" | {v:>8.2f}" if not (v != v) else f" | {'timeout':>8}"  # nan check
        for v in row
    )
    print(row_str)

print(f"{'='*72}")
print("  (мс, мінімум з 3 запусків — best performance)")
print("  timeout = пропущено (зайняло б > 30 сек)")

In [ ]:
# ── Візуалізація benchmark ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("NYC Taxi: Sorting Algorithm Benchmark", fontsize=14, fontweight='bold')

colors = {
    'Bubble Sort':    '#e74c3c',
    'Insertion Sort': '#e67e22',
    'Merge Sort':     '#3498db',
    'Quick Sort':     '#9b59b6',
    'Timsort':        '#27ae60',
}
markers = {
    'Bubble Sort': 'o', 'Insertion Sort': 's',
    'Merge Sort': '^',  'Quick Sort': 'D', 'Timsort': '*'
}

# ── Графік 1: всі алгоритми ───────────────────────────────────────────────
ax1 = axes[0]
for name, _ in algorithms:
    vals = bench_results[name]
    ns   = BENCH_SIZES
    # Фільтруємо nan
    pairs = [(n, v) for n, v in zip(ns, vals) if v == v]
    if pairs:
        xs, ys = zip(*pairs)
        ax1.plot(xs, ys,
                 color=colors[name],
                 marker=markers[name],
                 linewidth=2,
                 markersize=8,
                 label=name)

ax1.set_xlabel("Кількість поїздок (n)", fontsize=11)
ax1.set_ylabel("Час (мс)", fontsize=11)
ax1.set_title("Всі алгоритми", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Анотація для Bubble Sort
bubble_vals = [(n, v) for n, v in zip(BENCH_SIZES, bench_results['Bubble Sort']) if v == v]
if bubble_vals:
    last_n, last_v = bubble_vals[-1]
    ax1.annotate(
        'O(n^2) bottleneck',
        xy=(last_n, last_v),
        xytext=(last_n * 0.6, last_v * 0.8),
        fontsize=8,
        color=colors['Bubble Sort'],
        arrowprops=dict(arrowstyle='->', color=colors['Bubble Sort'], lw=1.2)
    )

# ── Графік 2: zoom без Bubble Sort ──────────────────────────────────────
ax2 = axes[1]
for name, _ in algorithms:
    if name == 'Bubble Sort':
        continue
    vals = bench_results[name]
    xs = BENCH_SIZES
    ax2.plot(xs, vals,
             color=colors[name],
             marker=markers[name],
             linewidth=2,
             markersize=8,
             label=name)

ax2.set_xlabel("Кількість поїздок (n)", fontsize=11)
ax2.set_ylabel("Час (мс)", fontsize=11)
ax2.set_title("Zoom: без Bubble Sort", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

timsort_5k = bench_results['Timsort'][-1]
ax2.text(0.05, 0.95,
         f"Timsort @ n=5000:\n{timsort_5k:.3f} ms\n(Baseline)",
         transform=ax2.transAxes,
         fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9))

plt.tight_layout()
plt.show()

In [ ]:
# ── Speedup table: відносна швидкість vs Timsort ──────────────────────────
print("=== Speedup vs Timsort ===")
print()
timsort_times = bench_results['Timsort']

print(f"{'Алгоритм':<18}" + "".join(f" | n={n:,}".rjust(12) for n in BENCH_SIZES))
print("-" * 70)

for name, _ in algorithms:
    if name == 'Timsort':
        continue
    row = []
    for i, n in enumerate(BENCH_SIZES):
        v = bench_results[name][i]
        t = timsort_times[i]
        if v != v or t == 0:   # nan
            row.append('skip')
        else:
            row.append(f"{v/t:.1f}x")
    print(f"{name:<18}" + "".join(f" | {r:>11}" for r in row))

print()
print("Висновок: Timsort = базова лінія. Будь-яка власна реалізація повільніша.")
print("Виняток: Insertion Sort для n < 20 (де Timsort САМ його викликає внутрішньо).")

---

## Частина 8: Dispatch Optimizer — Практичний Кейс

---

### Задача

Є нова заявка: клієнт в **зоні 132** (JFK Airport). Знайдіть 10 найближчих **і** найдешевших вільних машин, відсортованих за пріоритетом:

1. Спочатку — машини з тієї самої зони (мінімальний порожній пробіг)
2. Серед них — найкоротша відстань до клієнта (∼ proximity score)
3. При рівних зонах — найменша вартість поїздки (economy first)

### Чому це multi-key sort?

```python
key = lambda t: (
    0 if t['pu_zone'] == TARGET_ZONE else 1,  # зона: local=0, remote=1
    t['trip_distance'],                         # відстань
    t['fare_amount'],                           # тариф
)
```

Python `tuple` порівняння — лексикографічне: спочатку перший елемент, при рівності — другий, і т.д.

---

In [ ]:
# ── Dispatch Optimizer ────────────────────────────────────────────────────

TARGET_ZONE = 132   # JFK Airport area
TOP_N = 10

def dispatch_optimizer(all_trips, target_zone, top_n=10):
    """
    Знаходить та ранжує top_n поїздок для диспетчеризації.
    
    Пріоритет:
      1. Локальна зона (pu_zone == target_zone) йде першою
      2. Серед однієї зони -- менша відстань
      3. При рівних відстанях -- менший тариф
    
    Використовує Timsort через sorted() з tuple key -- O(n log n).
    """
    def priority_key(trip):
        zone_priority = 0 if trip['pu_zone'] == target_zone else 1
        return (zone_priority, trip['trip_distance'], trip['fare_amount'])

    # Timsort з multi-key -- O(n log n), стабільний
    ranked = sorted(all_trips, key=priority_key)
    return ranked[:top_n]


# ── Запуск ─────────────────────────────────────────────────────────────────
print(f"=== Dispatch Optimizer: нова заявка в зоні {TARGET_ZONE} ===")
print(f"    Шукаємо топ-{TOP_N} пріоритетних поїздок з {len(trips):,} доступних")
print()

t0 = time.perf_counter()
best_trips = dispatch_optimizer(trips, TARGET_ZONE, TOP_N)
t1 = time.perf_counter()

print(f"Час виконання: {(t1-t0)*1000:.2f} мс")
print()
print(f"{'#':<3} {'Zone':<6} {'Dist(mi)':<10} {'Fare($)':<9} {'Tip($)':<8} {'Пріоритет':<12}")
print("-" * 55)

for i, t in enumerate(best_trips):
    priority = "LOCAL" if t['pu_zone'] == TARGET_ZONE else "remote"
    print(f"{i+1:<3} {t['pu_zone']:<6} {t['trip_distance']:<10.2f} "
          f"{t['fare_amount']:<9.2f} {t['tip_amount']:<8.2f} {priority}")

local_count = sum(1 for t in best_trips if t['pu_zone'] == TARGET_ZONE)
print()
print(f"  Місцевих поїздок (zone={TARGET_ZONE}): {local_count}")
print(f"  Немісцевих:                          {TOP_N - local_count}")

# ── Benchmark: як швидко диспетчер може обробити черговий запит? ──────────
print()
rps = 1 / (timeit.timeit(
    lambda: dispatch_optimizer(trips, TARGET_ZONE, TOP_N),
    number=1000
) / 1000)
print(f"  Максимальна пропускна здатність: {rps:,.0f} запитів/сек")
print(f"  (при {len(trips):,} поїздок у пулі)")

---

## Частина 9: Аналітика — Топ Зони, Розподіл Тарифів

---

In [ ]:
# ── Аналітика на відсортованих даних ──────────────────────────────────────

# Сортуємо один раз — використовуємо для кількох аналізів
by_fare   = sorted(trips, key=lambda t: t['fare_amount'])
by_zone   = sorted(trips, key=lambda t: (t['pu_zone'], -t['total_amount']))
by_tip    = sorted(trips, key=lambda t: t['tip_amount'], reverse=True)

# Топ-5 зон за кількістю поїздок
zone_counts = defaultdict(int)
zone_revenue = defaultdict(float)
for t in trips:
    zone_counts[t['pu_zone']] += 1
    zone_revenue[t['pu_zone']] += t['total_amount']

top_zones = sorted(zone_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("=== Аналітика NYC Taxi Jan 2023 (вибірка 5,000) ===")
print()
print("Топ-10 зон посадки:")
print(f"{'Zone':<8} {'Trips':>7} {'Revenue($)':>12} {'Avg Revenue':>13}")
print("-" * 45)
for zone, count in top_zones:
    rev = zone_revenue[zone]
    avg = rev / count
    bar = '|' * (count // 5)
    print(f"{zone:<8} {count:>7,}  ${rev:>10,.2f}  ${avg:>10.2f}")

# Перцентилі тарифу
fares = [t['fare_amount'] for t in by_fare]
p25 = fares[len(fares) // 4]
p50 = fares[len(fares) // 2]
p75 = fares[len(fares) * 3 // 4]
p95 = fares[int(len(fares) * 0.95)]

print()
print("Розподіл тарифів (fare_amount):")
print(f"  Мін:  ${fares[1]:.2f}")
print(f"  P25:  ${p25:.2f}")
print(f"  P50:  ${p50:.2f}  (медіана)")
print(f"  P75:  ${p75:.2f}")
print(f"  P95:  ${p95:.2f}")
print(f"  Макс: ${fares[-1]:.2f}")

# Топ чайові
print()
print("Топ-3 поїздки за чайовими:")
for t in by_tip[:3]:
    tip_pct = t['tip_amount'] / t['fare_amount'] * 100 if t['fare_amount'] > 0 else 0
    print(f"  tip=${t['tip_amount']:.2f} ({tip_pct:.0f}% від тарифу)  "
          f"dist={t['trip_distance']:.1f}mi  zone {t['pu_zone']}→{t['do_zone']}")

In [ ]:
# ── Візуалізація аналітики ────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("NYC Yellow Taxi Jan 2023 — Sorting Analytics", fontsize=14, fontweight='bold')

# ── 1. Гістограма тарифів ────────────────────────────────────────────────
ax1 = axes[0][0]
p95_fare = fares[int(len(fares) * 0.95)]
clipped = [f for f in fares if f <= p95_fare]
ax1.hist(clipped, bins=50, color='#3498db', edgecolor='white', alpha=0.85)
ax1.axvline(p50, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Median: ${p50:.2f}')
ax1.axvline(sum(fares)/len(fares), color='#f39c12', linestyle='-.', linewidth=2,
            label=f'Mean: ${sum(fares)/len(fares):.2f}')
ax1.set_xlabel('Fare Amount ($)', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_title('Розподіл тарифів (fare_amount)', fontsize=10)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── 2. Топ-10 зон за поїздками ────────────────────────────────────────────
ax2 = axes[0][1]
zone_ids   = [str(z) for z, _ in top_zones]
zone_trips = [c for _, c in top_zones]
bars = ax2.bar(zone_ids, zone_trips, color='#27ae60', edgecolor='white', alpha=0.85)
ax2.bar_label(bars, fmt='%d', fontsize=8)
ax2.set_xlabel('Zone ID', fontsize=10)
ax2.set_ylabel('Trips Count', fontsize=10)
ax2.set_title('Топ-10 зон посадки', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# ── 3. Scatter: відстань vs тариф ────────────────────────────────────────
ax3 = axes[1][0]
sample_scatter = trips[::5]   # кожен 5-й запис для швидкості
distances = [t['trip_distance'] for t in sample_scatter if t['trip_distance'] < 30]
fares_sc  = [t['fare_amount']   for t in sample_scatter if t['trip_distance'] < 30]
ax3.scatter(distances, fares_sc, alpha=0.3, s=8, color='#9b59b6')
ax3.set_xlabel('Trip Distance (miles)', fontsize=10)
ax3.set_ylabel('Fare Amount ($)', fontsize=10)
ax3.set_title('Відстань vs Тариф', fontsize=10)
ax3.grid(True, alpha=0.3)

# ── 4. Benchmark bar chart ────────────────────────────────────────────────
ax4 = axes[1][1]
# Беремо результати для n=5000 (або останнє доступне значення)
algo_names = []
algo_times = []
for name, _ in algorithms:
    vals = bench_results[name]
    valid = [v for v in vals if v == v]
    if valid:
        algo_names.append(name)
        algo_times.append(valid[-1])

bar_colors = [colors[n] for n in algo_names]
bars4 = ax4.bar(range(len(algo_names)), algo_times, color=bar_colors, edgecolor='white', alpha=0.9)
ax4.bar_label(bars4, fmt='%.2f ms', fontsize=8)
ax4.set_xticks(range(len(algo_names)))
ax4.set_xticklabels([n.replace(' Sort', '') for n in algo_names], fontsize=8)
ax4.set_ylabel('Time (ms)', fontsize=10)
ax4.set_title(f'Benchmark (n={BENCH_SIZES[-1]:,}) — менше = краще', fontsize=10)
ax4.set_yscale('log')   # логарифмічна шкала — інакше Timsort не видно
ax4.grid(True, alpha=0.3, axis='y')
ax4.text(0.02, 0.97, 'Log scale', transform=ax4.transAxes,
         fontsize=8, color='gray', va='top')

plt.tight_layout()
plt.show()

---

## Частина 10: Фінальний Інсайт — Архітектурні Правила

---

### Ланцюжок вибору алгоритму

```
Є задача сортування
        ↓
Яка мова / рівень доступу?
        ↓
Python → ЗАВЖДИ sorted() / list.sort()  (Timsort в C)
        ↓  (тільки якщо є конкретна причина)
Потрібна власна реалізація?
        ├── n < 20?              → Insertion Sort
        ├── Стабільність?        → Merge Sort
        ├── Мінімум пам'яті?     → Quick Sort (in-place)
        ├── Гарантія O(n log n)? → Merge Sort або Heap Sort
        └── Embedded/OS kernel?  → Heap Sort
```

### Підсумкова таблиця

| Алгоритм | Best | Avg | Worst | Пам'ять | Stable | Практичне місце |
|---------|------|-----|-------|---------|--------|----------------|
| **Bubble** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ | Навчання. Ніде в prod |
| **Insertion** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ | n<20, всередині Timsort |
| **Merge** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | ✅ | Linked Lists, зовнішнє |
| **Quick** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | ❌ | C qsort, in-place |
| **Timsort** | $O(n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | ✅ | Python, Java, Android |

### Головний висновок

> **Алгоритм — це не лише математика. Це рішення про взаємодію з CPU та RAM.**
>
> Quicksort перемагає теоретично «кращий» Merge Sort на масивах,  
> тому що CPU кеш і лінійне сканування пам'яті — це реальна перевага.
>
> Але в Python — завжди `sorted()`. Timsort написаний на C і знає  
> про ваш CPU більше, ніж будь-яка реалізація на Python.

---

*Лабораторія 27 · Module 3 · Python Advanced · Viktor Nikoriak*

In [ ]:
# ── Бонус: інтерактивний слайдер ──────────────────────────────────────────
try:
    import ipywidgets as widgets
    from IPython.display import display

    output_w = widgets.Output()
    slider = widgets.IntSlider(
        value=500, min=100, max=min(3000, len(trips)),
        step=100, description='n trips:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    zone_dd = widgets.Dropdown(
        options=sorted(zone_counts.keys()),
        value=TARGET_ZONE,
        description='Target zone:',
        style={'description_width': 'initial'}
    )
    label_w = widgets.Label(value="")

    def on_change(change):
        n = slider.value
        zone = zone_dd.value
        sample = (trips * (n // len(trips) + 1))[:n]
        t_tim = timeit.timeit(
            lambda: dispatch_optimizer(sample, zone, 10),
            number=500
        ) / 500 * 1000
        t_qk = timeit.timeit(
            lambda: quick_sort_trips(sample, lambda t: t['trip_distance']),
            number=100
        ) / 100 * 1000
        rps_val = 1000 / t_tim if t_tim > 0 else 0
        label_w.value = (
            f"  n={n:,} | zone={zone} | "
            f"Timsort: {t_tim:.3f} ms | Quick: {t_qk:.2f} ms | "
            f"RPS: {rps_val:,.0f}"
        )

    slider.observe(on_change, names='value')
    zone_dd.observe(on_change, names='value')
    on_change(None)

    display(widgets.VBox([
        widgets.HTML("<b>Dispatch Optimizer: змінюйте n та зону</b>"),
        widgets.HBox([slider, zone_dd]),
        label_w
    ]))

except ImportError:
    print("ipywidgets не встановлений. pip install ipywidgets")
    print()
    print("Ручний тест dispatch_optimizer для кількох розмірів:")
    for n in [100, 500, 1000, 2000, len(trips)]:
        sample = (trips * (n // len(trips) + 1))[:n]
        t_ms = timeit.timeit(
            lambda s=sample: dispatch_optimizer(s, TARGET_ZONE, 10),
            number=500
        ) / 500 * 1000
        rps_val = 1000 / t_ms if t_ms > 0 else 0
        print(f"  n={n:>6,}: {t_ms:.3f} ms | {rps_val:>8,.0f} RPS")